# 02 · Regional FST and Implied Migrants

Loads Nick's pre-computed pairwise FST table (`table2.fst.xlsx`), converts to  
implied number of migrants (Nm = (1−FST) / 4·FST), and builds geographic distances  
between region centroids.

**Inputs**
- `table2.fst.xlsx` — Nick's pairwise FST (lower triangle) with 95% CIs (upper triangle)
- `suptable_merged.tsv` — for region centroid coordinates

**Outputs**
- `pairwise_nm.tsv`, `fst_matrix.tsv`
- `scatter_nm_vs_distance.pdf / .eps`
- `cleveland_nm_pairs.pdf / .eps`
- `heatmap_fst_nm.pdf / .eps`

## 1 · Imports and paths

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D
from scipy import stats
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.spatial.distance import squareform

sys.path.insert(0, str(Path("../scripts")))
from popgen_utils import fst_mat_to_nm_long, haversine_km

In [ ]:
PROJECT_DIR = Path("/Users/kidelu001/awi-work/science/projects/apal_popgen/")
OUT_DIR     = PROJECT_DIR / "output" / "tables"
PLOTS_DIR   = PROJECT_DIR / "output" / "plots" / "n_migrants"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

FST_XLSX     = OUT_DIR / "table2.fst.xlsx"
SUPTABLE_TSV = OUT_DIR / "suptable_merged.tsv"

## 2 · Parameters

In [ ]:
LAT_COL = "lat_clean"
LON_COL = "lon_clean"

# Rename Nick's two-letter codes to region names used in the metadata
REGION_RENAME = {
    "BH":  "Bahamas",
    "BZ":  "Belize",
    "CO":  "Colombia",
    "SA":  "Col_SA",
    "CB":  "Cuba",
    "CU":  "Curacao",
    "DR":  "DR",
    "FL":  "Florida",
    "HN":  "Honduras",
    "JM":  "Jamaica",
    "GOM": "Mex_Gulf",
    "MX":  "Mexico",
    "PR":  "PR",
    "VI":  "USVI",
}

# Map metadata region_sub values to Nick's region names where they differ
META_TO_NICK = {
    "Dominican Republic": "DR",
    "Puerto Rico":        "PR",
    "Mexico_Gulf":        "Mex_Gulf",
    "Colombia_SanAndres": "Col_SA",
}

REGION_GROUP_MAP = {
    "Bahamas":  "Northern",
    "Florida":  "Northern",
    "Mexico":   "MesoAm",
    "Belize":   "MesoAm",
    "Honduras": "MesoAm",
    "Mex_Gulf": "Mex_Gulf",
    "Cuba":               "Greater Antilles",
    "Jamaica":            "Greater Antilles",
    "DR":                 "Greater Antilles",
    "PR":                 "Greater Antilles",
    "USVI":               "Greater Antilles",
    "Curacao":   "Southern",
    "Colombia":  "Southern",
    "Col_SA":    "Col_SA",
}

PAIR_GROUP_COLORS = {
    "Northern":        "darkorange",
    "Greater Antilles":"cornflowerblue",
    "Southern":        "yellowgreen",
    "MesoAm":          "darkorchid",
    "Mex_Gulf":        "indianred",
    "Col_SA":          "forestgreen",
    "Cross-basin":     "grey",
}

## 3 · Load and parse Nick's FST table

In [ ]:
nick_raw = pd.read_excel(FST_XLSX, index_col=0)
nick_raw

In [ ]:
# Lower triangle = FST values; upper triangle = 95% CI strings; diagonal = 0
regions_raw = nick_raw.index.tolist()
fst_mat = pd.DataFrame(np.nan,  index=regions_raw, columns=regions_raw, dtype=float)
ci_mat  = pd.DataFrame("",      index=regions_raw, columns=regions_raw, dtype="string")

for i, r1 in enumerate(regions_raw):
    for j, r2 in enumerate(regions_raw):
        val = nick_raw.loc[r1, r2]
        if i == j:
            fst_mat.loc[r1, r2] = 0.0
        elif i > j:   # lower triangle: FST
            v = pd.to_numeric(val, errors="coerce")
            fst_mat.loc[r1, r2] = v
            fst_mat.loc[r2, r1] = v
        else:         # upper triangle: CI string
            ci_mat.loc[r1, r2] = "" if pd.isna(val) else str(val).replace("\n", " ")

fst_mat = fst_mat.rename(index=REGION_RENAME, columns=REGION_RENAME)
ci_mat  = ci_mat.rename(index=REGION_RENAME,  columns=REGION_RENAME)

fst_mat

## 4 · Region centroid coordinates

In [ ]:
meta = pd.read_csv(SUPTABLE_TSV, sep="\t", low_memory=False)

region_coords = (
    meta.groupby("region_sub", dropna=False)
    .agg(lat=(LAT_COL, "median"), lon=(LON_COL, "median"))
    .reset_index()
    .rename(columns={"region_sub": "region"})
)

# Align region names to Nick's convention
region_coords["region"] = region_coords["region"].replace(META_TO_NICK)
region_coords

## 5 · Compute Nm and geographic distances

In [ ]:
nm_long = fst_mat_to_nm_long(
    fst_mat,
    region_coords,
    region_col="region",
    lat_col="lat",
    lon_col="lon",
    pair_group_map=REGION_GROUP_MAP,
)
nm_long.head(10)

In [ ]:
# Symmetric geographic distance matrix
geo_mat = pd.DataFrame(np.nan, index=fst_mat.index, columns=fst_mat.columns, dtype=float)
coord_lookup = region_coords.set_index("region")

for r1 in fst_mat.index:
    for r2 in fst_mat.columns:
        if r1 == r2:
            geo_mat.loc[r1, r2] = 0.0
        else:
            geo_mat.loc[r1, r2] = haversine_km(
                coord_lookup.loc[r1, "lat"], coord_lookup.loc[r1, "lon"],
                coord_lookup.loc[r2, "lat"], coord_lookup.loc[r2, "lon"],
            )

## 6 · Hierarchical ordering

In [ ]:
# Order regions by average-linkage clustering on FST
cluster_src = fst_mat.copy()
np.fill_diagonal(cluster_src.values, 0.0)
Z     = linkage(squareform(cluster_src.to_numpy(), checks=False), method="average")
order = leaves_list(Z)
ordered_regions = fst_mat.index[order].tolist()

fst_ord = fst_mat.loc[ordered_regions, ordered_regions]
geo_ord = geo_mat.loc[ordered_regions, ordered_regions]
nm_mat  = pd.DataFrame(np.nan, index=fst_mat.index, columns=fst_mat.columns, dtype=float)
for r1 in fst_mat.index:
    for r2 in fst_mat.columns:
        fst = fst_mat.loc[r1, r2]
        if r1 != r2 and pd.notna(fst) and 0 < fst < 1:
            nm_mat.loc[r1, r2] = (1 - fst) / (4 * fst)
nm_ord = nm_mat.loc[ordered_regions, ordered_regions]

In [ ]:
# Reference table: Nm (upper triangle) vs geographic distance km (lower triangle)
ref = pd.DataFrame("", index=ordered_regions, columns=ordered_regions, dtype="string")
for i, r1 in enumerate(ordered_regions):
    for j, r2 in enumerate(ordered_regions):
        if i < j:
            v = nm_ord.loc[r1, r2]
            ref.loc[r1, r2] = "" if pd.isna(v) else f"{v:.0f}"
        elif i > j:
            v = geo_ord.loc[r1, r2]
            ref.loc[r1, r2] = "" if pd.isna(v) else f"{v:.0f}"
        else:
            ref.loc[r1, r2] = "—"
ref

## 7 · Figures

### 7a · Scatter: log10 Nm vs geographic distance

In [ ]:
def plot_nm_vs_distance(
    plot_df,
    y_col="log10_n_migrants",
    label_sd_mult=1.5,
    label_side="both",
    label_offsets=None,
    figsize=(7, 7),
    save_base=None,
):
    """
    Scatter of implied Nm vs distance with OLS fit.
    Points whose residuals exceed label_sd_mult × SD are labelled.
    label_side : 'both' | 'above' | 'below'
    label_offsets : dict keyed by pair string, values: {dx, dy, ha, va}
    """
    if label_offsets is None:
        label_offsets = {}
    default = {"dx": 8, "dy": 0.0, "ha": "left", "va": "center"}

    df   = plot_df[np.isfinite(plot_df["geo_dist_km"]) & np.isfinite(plot_df[y_col])].copy()
    x, y = df["geo_dist_km"].values, df[y_col].values
    ols  = stats.linregress(x, y)

    df["pred"]  = ols.slope * x + ols.intercept
    df["resid"] = y - df["pred"].values
    resid_sd    = df["resid"].std()
    cutoff      = label_sd_mult * resid_sd

    if label_side == "both":
        label_df = df[df["resid"].abs() >= cutoff]
    elif label_side == "above":
        label_df = df[df["resid"] >= cutoff]
    else:
        label_df = df[df["resid"] <= -cutoff]

    fig, ax = plt.subplots(figsize=figsize)
    for group, sub in df.groupby("pair_group"):
        ax.scatter(sub["geo_dist_km"], sub[y_col],
                   label=group, alpha=0.85, s=80,
                   c=PAIR_GROUP_COLORS.get(group, "#888888"),
                   edgecolors="black", linewidths=0.8)

    xline = np.linspace(x.min(), x.max(), 200)
    ax.plot(xline, ols.slope * xline + ols.intercept, color="grey", lw=1)

    for _, row in label_df.iterrows():
        style = {**default, **label_offsets.get(row["pair"], {})}
        ax.text(row["geo_dist_km"] + style["dx"], row[y_col] + style["dy"],
                row["pair"], fontsize=8, alpha=0.9,
                ha=style["ha"], va=style["va"])

    ax.set_xlabel("Geographic distance (km)")
    ax.set_ylabel("log$_{10}$ implied N$_m$" if "log10" in y_col else "Implied N$_m$")
    ax.set_title("Region-pair connectivity vs geographic distance")
    ax.legend(frameon=False, fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_base is not None:
        for ext in ("pdf", "eps"):
            fig.savefig(f"{save_base}.{ext}", bbox_inches="tight")
        print(f"Saved → {save_base}.pdf / .eps")
    plt.show()
    return {"ols": ols, "r2": ols.rvalue**2}

In [ ]:
# Adjust to avoid label overlap
LABEL_OFFSETS = {
    "Belize – Honduras":   {"dx": 21, "dy":  0.02, "ha": "left", "va": "bottom"},
    "Honduras – Mexico":   {"dx": 21, "dy":  0.02, "ha": "left", "va": "bottom"},
    "Belize – Mexico":     {"dx": 21, "dy": -0.04, "ha": "left", "va": "top"},
    "Florida – Mexico":    {"dx": 21, "dy":  0.02, "ha": "left", "va": "bottom"},
    "Belize – Florida":    {"dx": 21, "dy":  0.02, "ha": "left", "va": "bottom"},
    "Florida – Honduras":  {"dx": 21, "dy": -0.04, "ha": "left", "va": "top"},
    "Colombia – Curacao":  {"dx": 21, "dy": -0.06, "ha": "left", "va": "top"},
}

plot_nm_vs_distance(
    nm_long,
    y_col="log10_n_migrants",
    label_sd_mult=1.5,
    label_side="both",
    label_offsets=LABEL_OFFSETS,
    save_base=PLOTS_DIR / "scatter_nm_vs_distance",
)

### 7b · Cleveland dot plot: log10 Nm per region pair

In [ ]:
plot_df = (
    nm_long[nm_long["log10_n_migrants"].notna()]
    .assign(pair_scope=lambda d: np.where(d["group1"] == d["group2"], "within", "between"))
    .sort_values("log10_n_migrants")
    .reset_index(drop=True)
)

scope_style = {
    "within":  {"facecolors": "dimgrey",   "edgecolors": "dimgrey"},
    "between": {"facecolors": "whitesmoke","edgecolors": "dimgrey"},
}

fig, ax = plt.subplots(figsize=(8, 12))
for _, row in plot_df.iterrows():
    style = scope_style.get(row["pair_scope"], {"facecolors": "grey", "edgecolors": "dimgrey"})
    ax.scatter(row["log10_n_migrants"], row["pair"],
               linewidths=0.8, alpha=0.9, **style)

ax.set_xlabel("log$_{10}$ implied N$_m$")
ax.set_ylabel("")
ax.set_title("Region-pair estimated number of migrants")
ax.grid(True, axis="x", alpha=0.3)
ax.tick_params(axis="y", labelsize=6)

legend_handles = [
    Line2D([0], [0], marker="o", linestyle="", markerfacecolor="dimgrey",
           markeredgecolor="dimgrey", label="within group", markersize=7),
    Line2D([0], [0], marker="o", linestyle="", markerfacecolor="whitesmoke",
           markeredgecolor="dimgrey", label="between groups", markersize=7),
]
ax.legend(handles=legend_handles, frameon=False)
plt.tight_layout()

for ext in ("pdf", "eps"):
    fig.savefig(str(PLOTS_DIR / f"cleveland_nm_pairs.{ext}"), bbox_inches="tight")
print(f"Saved → {PLOTS_DIR}/cleveland_nm_pairs.pdf / .eps")
plt.show()

### 7c · Heatmap: FST (lower triangle) / log10 Nm (upper triangle)

In [ ]:
log10_nm_ord = np.log10(nm_ord.replace(0, np.nan))

n          = len(fst_ord)
mask_upper = np.tril(np.ones((n, n), dtype=bool), k=0)   # hides upper for FST layer
mask_lower = np.triu(np.ones((n, n), dtype=bool), k=0)   # hides lower for Nm layer

fst_vmin, fst_vmax = np.nanmin(fst_ord.to_numpy()),        np.nanmax(fst_ord.to_numpy())
nm_vmin,  nm_vmax  = np.nanmin(log10_nm_ord.to_numpy()),   np.nanmax(log10_nm_ord.to_numpy())

cmap_fst = mpl.colormaps["mako_r"]
cmap_nm  = mpl.colormaps["viridis"]
norm_fst = mpl.colors.Normalize(vmin=fst_vmin, vmax=fst_vmax)
norm_nm  = mpl.colors.Normalize(vmin=nm_vmin,  vmax=nm_vmax)

fig  = plt.figure(figsize=(10.5, 8.5))
ax      = fig.add_axes([0.24, 0.14, 0.56, 0.56])
cax_nm  = fig.add_axes([0.12, 0.28, 0.018, 0.28])
cax_fst = fig.add_axes([0.84, 0.28, 0.018, 0.28])

sns.heatmap(fst_ord,        mask=mask_upper, cmap=cmap_fst, norm=norm_fst,
            square=True, linewidths=0.5, linecolor="white", cbar=False, ax=ax)
sns.heatmap(log10_nm_ord,   mask=mask_lower, cmap=cmap_nm,  norm=norm_nm,
            square=True, linewidths=0.5, linecolor="white", cbar=False, ax=ax)

cb_nm  = fig.colorbar(mpl.cm.ScalarMappable(cmap=cmap_nm,  norm=norm_nm),  cax=cax_nm,  orientation="vertical")
cb_fst = fig.colorbar(mpl.cm.ScalarMappable(cmap=cmap_fst, norm=norm_fst), cax=cax_fst, orientation="vertical")
cb_nm.set_label("log$_{10}$ implied N$_m$", rotation=90, labelpad=10)
cb_fst.set_label("$F_{ST}$",                rotation=90, labelpad=10)
cb_nm.ax.tick_params(labelsize=8)
cb_fst.ax.tick_params(labelsize=8)

ax.set_title("Pairwise $F_{ST}$ (lower) · log$_{10}$ implied N$_m$ (upper)", pad=18)
ax.tick_params(axis="x", rotation=90)
ax.tick_params(axis="y", rotation=0)
plt.tight_layout()

for ext in ("pdf", "eps"):
    fig.savefig(str(PLOTS_DIR / f"heatmap_fst_nm.{ext}"), bbox_inches="tight")
print(f"Saved → {PLOTS_DIR}/heatmap_fst_nm.pdf / .eps")
plt.show()

## 8 · Save tables

In [ ]:
nm_long.to_csv(OUT_DIR / "pairwise_nm.tsv",  sep="\t", index=False)
fst_mat.to_csv(OUT_DIR / "fst_matrix.tsv",   sep="\t")

print(f"Saved → {OUT_DIR}/pairwise_nm.tsv")
print(f"Saved → {OUT_DIR}/fst_matrix.tsv")